# Housing Price Prediction

Train and evaluate regression models using the local `housing.csv` dataset. Run the code cells in order.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score



In [ ]:
# Load the dataset
housing = pd.read_csv("housing.csv")

# Create a stratified testset
housing["income_cat"] = pd.cut(housing["median_income"], bins=[0,1.5,3.0,4.5,6.0, np.inf], labels=[1,2,3,4,5])


split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index].drop("income_cat", axis=1)
    strat_test_set = housing.loc[test_index].drop("income_cat", axis=1)



In [ ]:
# work on copy of training data
housing = strat_train_set.copy()

# Seprate labels or features
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)

# print(housing, housing_labels)

# Seprate numerical and catogarical columns
num_attribs = housing.drop("ocean_proximity", axis=1).columns.tolist()
cat_attribs = ["ocean_proximity"]

# Make pipelines for both
num_pipeline = Pipeline([
    ("imputes", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore")) 
])

# Construct Full Pipeline
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

# Transform the data
housing_prepared = full_pipeline.fit_transform(housing)

# print(housing_prepared.shape)



In [ ]:
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

# Decision Tree
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(housing_prepared, housing_labels)

# Random Forest
forest_reg = RandomForestRegressor(random_state=42)
forest_reg.fit(housing_prepared, housing_labels)

# predict using traning Data
lin_pred = lin_reg.predict(housing_prepared)
tree_pred = tree_reg.predict(housing_prepared)
forest_pred = forest_reg.predict(housing_prepared)

# Calculate RMSE
lin_rmse = mean_squared_error(housing_labels, lin_pred)
tree_rmse = mean_squared_error(housing_labels, tree_pred)
forest_rmse = mean_squared_error(housing_labels, forest_pred)

print("Linear Regression RMSE:", lin_rmse)
print("Decision Tree RMSE:", tree_rmse)
print("Random Forest RMSE:", forest_rmse)



In [ ]:
tree_rmses = -cross_val_score(tree_reg, housing_prepared, housing_labels, scoring="neg_root_mean_squared_error", cv=10)

print("Decision Tree CV RMSEs: ", tree_rmses)
print("Cross Validation Performance: ")
print(pd.Series(tree_rmses).describe())